In [1]:
import requests
import json

url = "https://api.binance.com/api/v3/ticker/24hr"

try:
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    all_coins = response.json()

    # Select 10 popular USDT trading pairs
    symbols = [
        "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
        "ADAUSDT", "DOGEUSDT", "TRXUSDT", "AVAXUSDT", "LINKUSDT"
    ]

    crypto_data = [
        coin for coin in all_coins
        if coin["symbol"] in symbols
    ]

    if not crypto_data:
        print("No cryptocurrency data found.")
    else:
        with open("crypto_data.json", "w", encoding="utf-8") as file:
            json.dump(crypto_data, file, indent=4)

        print("Data saved to crypto_data.json")
        print("Coins fetched:", len(crypto_data))

except requests.RequestException as e:
    print("Unable to fetch Binance data:", e)

Data saved to crypto_data.json
Coins fetched: 10


In [2]:
import json

def find_most_volatile_coin(data):
    return max(
        data,
        key=lambda coin: abs(float(coin["priceChangePercent"]))
    )["symbol"]

with open("crypto_data.json", "r", encoding="utf-8") as file:
    data = json.load(file)

coin = find_most_volatile_coin(data)

print("Most volatile coin:", coin)

for item in data:
    if item["symbol"] == coin:
        print("24h Price Change:", item["priceChangePercent"], "%")
        break

Most volatile coin: ADAUSDT
24h Price Change: -2.446 %


In [3]:
import json

with open("crypto_data.json", "r", encoding="utf-8") as file:
    data = json.load(file)

prices = [float(coin["lastPrice"]) for coin in data]

average_price = sum(prices) / len(prices)

print("Average price:", average_price)
print("\nCoins below average price:")

for coin in data:
    price = float(coin["lastPrice"])

    if price < average_price:
        print(coin["symbol"], "-", price)

Average price: 8506.235989

Coins below average price:
ETHUSDT - 2406.97
BNBUSDT - 739.93
ADAUSDT - 0.2154
XRPUSDT - 1.3195
TRXUSDT - 0.3295
LINKUSDT - 12.174
DOGEUSDT - 0.07949
SOLUSDT - 104.09
AVAXUSDT - 9.252


In [4]:
import json

def rank_coins_by_volume(data):
    sorted_coins = sorted(
        data,
        key=lambda coin: float(coin["quoteVolume"]),
        reverse=True
    )

    print("Top 5 Coins by 24h Trading Volume")

    for rank, coin in enumerate(sorted_coins[:5], start=1):
        volume = float(coin["quoteVolume"])
        print(f"{rank}. {coin['symbol']} - Volume: ${volume:,.2f}")

with open("crypto_data.json", "r", encoding="utf-8") as file:
    data = json.load(file)

rank_coins_by_volume(data)

Top 5 Coins by 24h Trading Volume
1. BTCUSDT - Volume: $1,465,454,459.81
2. ETHUSDT - Volume: $744,663,798.75
3. SOLUSDT - Volume: $170,614,163.10
4. XRPUSDT - Volume: $105,410,181.17
5. BNBUSDT - Volume: $66,632,478.68


In [5]:
import requests
import json
import time
import schedule
from datetime import datetime

URL = "https://api.binance.com/api/v3/ticker/24hr"

def fetch_crypto_data():
    for attempt in range(5):
        try:
            response = requests.get(URL, timeout=20)

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    wait = float(retry_after)
                else:
                    wait = 2 ** attempt

                print(f"Rate limit reached. Retrying in {wait} seconds.")
                time.sleep(wait)
                continue

            response.raise_for_status()
            all_coins = response.json()

            symbols = {
                "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT",
                "XRPUSDT", "ADAUSDT", "DOGEUSDT", "TRXUSDT",
                "AVAXUSDT", "LINKUSDT"
            }

            data = [
                coin for coin in all_coins
                if coin["symbol"] in symbols
            ]

            if not data:
                print("No matching coin data returned.")
                return

            with open("crypto_data.json", "w", encoding="utf-8") as file:
                json.dump(data, file, indent=4)

            print("Updated:", datetime.now().strftime("%d-%m-%Y %H:%M:%S"))

            most_volatile = max(
                data,
                key=lambda coin: abs(float(coin["priceChangePercent"]))
            )

            print("Most volatile:", most_volatile["symbol"])
            print("24h change:", most_volatile["priceChangePercent"], "%")

            return

        except requests.RequestException as e:
            print(f"Request failed (attempt {attempt + 1}/5): {e}")

            if attempt < 4:
                time.sleep(2 ** attempt)

        except (ValueError, KeyError, TypeError) as e:
            print("Error processing API data:", e)
            return

    print("Unable to update crypto data after multiple attempts.")

def job():
    print("\nFetching cryptocurrency data...")
    fetch_crypto_data()

# Run once immediately
job()

# Repeat every hour
schedule.every(1).hour.do(job)

print("Crypto tracker running. Press Ctrl+C to stop.")

while True:
    try:
        schedule.run_pending()
        time.sleep(1)
    except KeyboardInterrupt:
        print("Crypto tracker stopped.")
        break


Fetching cryptocurrency data...
Updated: 23-10-2026 09:54:30
Most volatile: ADAUSDT
24h change: -2.535 %
Crypto tracker running. Press Ctrl+C to stop.
Crypto tracker stopped.
